# 4 — Complete the remaining 448 new episodes
Run only after smoke approval. ID and OOD run sequentially on the same physical A100. Each process is detached and resumable. Start ID, wait for it to finish, then start OOD—never both simultaneously.


In [ ]:
import os, subprocess, sys, time
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; OUT=Path.home()/"stage1"; GPU=(Path.home()/"stage1_gpu.txt").read_text().strip(); P=Path.home()/"LIBERO-plus"
def launch(scene):
    pidfile=OUT/f"stage1_{scene}.pid"; log=OUT/f"stage1_{scene}.log"
    if pidfile.exists():
        old=int(pidfile.read_text());
        try: os.kill(old,0); raise RuntimeError(f"existing {scene} process {old} is alive")
        except ProcessLookupError: pass
    py=Path.home()/("venv-stage1-id/bin/python" if scene=='id' else "venv-stage1-ood/bin/python")
    env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","PYTHONUNBUFFERED":"1"})
    if scene=='ood': env['PYTHONPATH']=str(P)
    cmd=[str(py),"-u","-m","async_vla_benchmark.scripts.run_stage1","--config",str(R/"async_vla_benchmark/configs/stage1.yaml"),"--manifest",str(OUT/"stage1_manifest.csv"),"--output-dir",str(OUT),"--scene",scene,"--resume","--verbose"]
    fh=open(log,"ab"); proc=subprocess.Popen(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,start_new_session=True); pidfile.write_text(str(proc.pid)); print("launched",scene,proc.pid,log)
# First invocation: launch('id'). After progress reports ID complete and its PID is dead, invoke launch('ood').
launch('id')


In [ ]:
# Re-run this progress cell any time, including after reconnecting.
import csv
manifest=list(csv.DictReader(open(OUT/"stage1_manifest.csv"))); planned_new={r['run_id'] for r in manifest if r['reuse_stage0'].lower()!='true'}
done={p.stem for p in (OUT/"episodes").glob("*.json")}; print(f"new episodes {len(done & planned_new)}/456; remaining {456-len(done & planned_new)}")
for scene in ('id','ood'):
    pf=OUT/f"stage1_{scene}.pid"; alive=False
    if pf.exists():
        try: os.kill(int(pf.read_text()),0); alive=True
        except (ProcessLookupError,PermissionError): pass
    log=OUT/f"stage1_{scene}.log"; print(scene,"alive=",alive); print(''.join(log.read_text().splitlines(True)[-8:]) if log.exists() else '(not started)')
print("Checkpoint the stage1 directory off-machine regularly; do not wait for shutdown day.")


In [ ]:
# Run this only after ID reports dead and all 36 new ID artifacts exist.
rows=list(csv.DictReader(open(OUT/"stage1_manifest.csv"))); id_new={r['run_id'] for r in rows if r['scene_condition']=='id' and r['reuse_stage0'].lower()!='true'}; done={p.stem for p in (OUT/"episodes").glob('*.json')}
if len(id_new & done)!=36: raise SystemExit(f"STOP: new ID is {len(id_new & done)}/36")
launch('ood')
